# Contrastive Learning for Retrieval: InfoNCE, Temperature, and Negative Sampling

This is the **narrative half** of the topic's notebook pillar. It imports the canonical,
tested module `infonce_contrastive_objective.py` — which *owns the numbers* — and walks the
topic section by section, running the verification harness so each pedagogical claim renders
as executed output. The `.py` is the regression test; this notebook is the story.

**Run:** `uv run --with numpy --with scipy --with jupyter jupyter execute notebooks/infonce-contrastive-objective/01_infonce_contrastive_objective.ipynb`

We read one loss four ways: the objective itself, a mutual-information lower bound, the
alignment/uniformity decomposition on the hypersphere, and the temperature-sharpened
hard-negative gradient.

In [ ]:
import sys, pathlib

# Import the canonical module whether we are run from the repo root or this directory.
here = pathlib.Path.cwd()
for cand in (here, here / "notebooks" / "infonce-contrastive-objective",
             here.parent, here / "notebooks"):
    p = cand / "infonce_contrastive_objective.py"
    if p.exists():
        sys.path.insert(0, str(cand)); break

import infonce_contrastive_objective as nce
print("loaded", nce.__file__)

## 1. The contrastive setup

A query $q$, its positive document $d^+$, and a batch of negatives $\{d_i^-\}$ are all
L2-normalized, so the score $s = \langle \hat q, \hat d\rangle / \tau$ is a
temperature-scaled cosine — the very similarity the retrieval-problem topic took as given,
now turned into a training signal. Positives and negatives are modeled as draws from
von Mises–Fisher clusters on the sphere (the hypersphere topic's law), so this module
*imports* its prerequisites rather than reinventing the geometry.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
d, N = 16, 7
z_q = nce.normalize(rng.standard_normal(d))
z_pos = nce.normalize(rng.standard_normal(d))
Z_neg = nce.normalize(rng.standard_normal((N, d)))
print("score(q, d+)      =", round(float(nce.scores(z_q, z_pos[None, :])[0]), 4))
print("scores(q, negs)   =", np.round(nce.scores(z_q, Z_neg), 4))

## 2. The InfoNCE objective

InfoNCE is the $(N{+}1)$-way cross-entropy that must pick the positive out of the batch:

$$\mathcal{L} = -\,\mathbb{E}\left[\log\frac{e^{\langle\hat q,\hat d^+\rangle/\tau}}
{e^{\langle\hat q,\hat d^+\rangle/\tau} + \sum_i e^{\langle\hat q,\hat d_i^-\rangle/\tau}}\right].$$

It is invariant to a constant shift of all scores; as $\tau\to\infty$ it degenerates to the
uniform classifier value $\log(N{+}1)$; and a positive far above the negatives drives it to
zero. The in-batch form treats every other row's positive as a negative.

In [ ]:
nce.test_loss_is_cross_entropy_and_invariances()
nce.test_batch_matches_single()

## 3. Movement 1 — InfoNCE is a mutual-information lower bound (CPC)

van den Oord, Li & Vinyals (2018): minimizing InfoNCE maximizes a lower bound on the mutual
information between query and positive,

$$I(q; d^+) \;\ge\; \log(N{+}1) - \mathcal{L}.$$

The bound is **ceilinged at $\log(N{+}1)$**, so on a finite batch it *saturates*: more
negatives raise the ceiling, which is the honest reason large batches help. We verify it on
a Gaussian joint where $I = -\tfrac12\ln(1-\rho^2)$ is exact, using the Bayes-optimal critic
so the test isolates the bound itself, not an encoder.

In [ ]:
nce.test_mi_lower_bound_holds_and_saturates()
for r in nce.mi_bound_curve(nce.MI_TARGET_NATS, nce.N_NEG_GRID, n_batches=20000, seed=0):
    print(r)

## 4. Movement 2 — alignment and uniformity on the sphere (Wang & Isola)

As the negative count grows, the loss splits into **alignment** (positives pulled together,
$\mathbb{E}\,\lVert\hat q-\hat d^+\rVert^2$) and **uniformity** (every embedding pushed
toward the uniform distribution on $S^{d-1}$, $\log\mathbb{E}\,e^{-t\lVert\hat x-\hat y\rVert^2}$).
The uniformity optimum is the *exact* uniform sphere law of the hypersphere topic — the
$\kappa\to 0$ limit of vMF — and temperature acts as an inverse concentration $\tau\sim 1/\kappa$.
Minimizing alignment+uniformity directly and minimizing InfoNCE land at the same
configuration.

In [ ]:
nce.test_uniformity_minimized_by_uniform_sphere()
nce.test_alignment_uniformity_decomposition()
nce.test_temperature_tradeoff_directions()

## 5. Movement 3 — temperature and the hard-negative gradient

The gradient with respect to the scores is exact and elementary:

$$\frac{\partial\mathcal L}{\partial s^+} = -\frac{1-p^+}{\tau}, \qquad
\frac{\partial\mathcal L}{\partial s_i^-} = +\frac{p_i}{\tau}, \qquad p = \mathrm{softmax}(s/\tau).$$

So the gradient is a **softmax-weighted repulsion**: the nearest (hardest) negative dominates,
and $\tau$ controls how sharply — small $\tau$ concentrates almost all the push on a single
negative (entropy of the weights $\to 0$), large $\tau$ spreads it uniformly (entropy
$\to \log N$).

In [ ]:
nce.test_gradient_structure()
nce.test_temperature_concentrates_gradient()

## 6. Finance case study — hard negatives are same-sector documents

Fine-tuning a financial dual encoder with in-batch-negative InfoNCE: each sector is a vMF
cluster, each company a tight sub-cluster, a query and its positive are two documents from
the same company, and the **hard negatives are same-sector, different-company documents** —
geometrically near but irrelevant. At small temperature the gradient routes overwhelmingly
onto those same-sector hard negatives. (Synthetic vMF stand-in, not a trained encoder.)

In [ ]:
nce.finance_demo()
nce.test_finance_hard_negatives_dominate()

## 7. The numbers the laboratory mirrors

`viz_constants()` prints exactly what `InfoNCEContrastiveLaboratory.tsx` bakes to the decimal
— the gradient-weight geometry, the MI-bound curve against its ceiling, the
alignment/uniformity convergence, and the finance hard-negative share. The TypeScript
recomputes only closed forms (softmax weights, $\log(N{+}1)$, $\kappa\sim 1/\tau$); every
*measured* number is baked from here.

In [ ]:
nce.viz_constants()
print("All checks passed.")